# The Language Model Backend

A generative backend for krrood's Entity Query Language that resolves underspecified
`Match` queries by asking a language model.

The point of this notebook is that **almost none of it needs a language model**. Reflection,
prompt assembly, decoding, write-back and validation are the bulk of the package, and all of
them run offline with a `ScriptedClient`. The real client goes in at the end, and nothing else changes.

Three questions frame each section:

- *What does the backend know?* -- a set of typed, path-named slots and the query's constraints. Nothing else.
- *Where does world knowledge enter?* -- through two injected seams, never through an import.
- *What can it be trusted to do?* -- generate values, never mint individuals.

## 0. Setup

Run from the repository root, with `krrood` importable.

```bash
pip install -e .
```

In [ ]:
import json
import sys
import warnings

warnings.filterwarnings("ignore")
sys.path.insert(0, "../src")
sys.path.insert(0, "..")

from krrood.entity_query_language.factories import a, variable

from examples.kitchen_domain import (
    ApproachDirection,
    Container,
    Grasp,
    Gripper,
    PickUp,
    VerticalAlignment,
    build_kitchen,
)
from llm_reasoner import (
    LanguageModelBackend,
    ScriptedClient,
    SchemaOnlyContextProvider,
    StaticContextProvider,
    SlotKind,
    reflect_slots,
)
from llm_reasoner.exceptions import (
    NoCandidateReferents,
    NoValidCandidates,
    ReferentSlotHasNoDomain,
    RepairBudgetExhausted,
)

kitchen = build_kitchen()
kitchen

## 1. An underspecified query

`...` marks a slot the caller does not know. `variable(Gripper)` marks a slot whose value must
be *selected* from things that exist rather than invented -- and with no domain argument, krrood
takes the extension from its `SymbolGraph`, because `Gripper` subclasses `Symbol` and every
instance registers itself on construction.

> **Gotcha.** The `SymbolGraph` holds its members weakly. An individual nothing else references
> is collected and vanishes from `variable(T)`'s extension. `build_kitchen()` keeps a strong
> reference for exactly this reason; in a real system the world holds it.

In [ ]:
def underspecified_grasp():
    """A fresh Match. Build a new one per experiment -- a Match resolves once."""
    match = a(Grasp)(
        approach_direction=...,
        vertical_alignment=...,
        gripper=variable(Gripper),
        rotate_gripper=...,
        manipulation_offset=...,
    )
    match.where(match.variable.manipulation_offset > 0.0)
    return match


match = underspecified_grasp()
match

### A binding trap worth knowing

Conditions must be bound through `match.variable`. Binding through `match.expression` produces
a condition that compiles, verbalizes, and then silently matches nothing -- with the shipped
`EntityQueryLanguageGenerativeBackend` too, so it is not specific to this package.

The cell below demonstrates it: same query, same constraint, 12 results versus 0.

In [ ]:
from krrood.entity_query_language.backends import EntityQueryLanguageGenerativeBackend

deterministic = EntityQueryLanguageGenerativeBackend()


def enumerable(bind_through):
    m = a(Grasp)(
        approach_direction=...,
        vertical_alignment=...,
        gripper=variable(Gripper),
        rotate_gripper=variable(bool, [True, False]),
        manipulation_offset=variable(float, [0.05, -0.02]),
    )
    m.where(bind_through(m).manipulation_offset > 0.0)
    return m


print("bound via match.variable  :", len(list(deterministic.evaluate(enumerable(lambda m: m.variable)))))
print("bound via match.expression:", len(list(deterministic.evaluate(enumerable(lambda m: m.expression)))))

## 2. Reflection: what the backend actually knows

Everything the model is later told is derived here, from the query alone -- type annotations for
the schema, PEP 257 attribute docstrings for the meaning, dotted access paths for the vocabulary.
No world, no robot, no domain package.

Each slot's *kind* is the epistemic classification:

| kind | what is missing | what is allowed |
|---|---|---|
| `VALUE` | a self-identifying value | generation |
| `REFERENT` | an individual | selection from an extension, never invention |
| `BOUND` | nothing | used as context, never asked about |

In [ ]:
for slot in reflect_slots(underspecified_grasp()):
    extra = ""
    if slot.members:
        extra = " | ".join(m.name for m in slot.members)
    elif slot.domain is not None:
        extra = f"{len(slot.domain)} candidate(s)"
    print(f"{slot.kind.name:9} {slot.path:34} {slot.type_name:18} {extra}")
    print(f"{'':9} {slot.docstring}")

Nesting flattens into dotted paths, and the nesting node itself is not a slot. This is what
lets one prompt cover a whole designator tree.

In [ ]:
nested = a(PickUp)(
    target=variable(Container),
    grasp=a(Grasp)(
        approach_direction=...,
        vertical_alignment=...,
        gripper=variable(Gripper),
        rotate_gripper=...,
        manipulation_offset=...,
    ),
    lift_height=...,
)

for slot in reflect_slots(nested):
    print(f"{slot.kind.name:9} {slot.path}")

## 3. The prompt

`describe()` assembles the prompt without calling anything, so you can iterate on wording
for free. Three layers:

1. **Schema** -- reflected. Always available, zero dependencies.
2. **The request** -- the whole query verbalized by krrood. Unfilled leaves render as the
   literal word *Ellipsis*; the system prompt glosses it as a blank to fill.
3. **Situation** -- from the context provider. The only world-aware layer.

Note the referent slot: candidates appear as **integer tokens**. The model never receives an
object, only a description addressed by a token, which is how the resolved value can be a live
world individual while this package stays decoupled. It is the same trick the probabilistic
backend plays with hashes.

In [ ]:
offline = LanguageModelBackend(client=ScriptedClient([]))
print(offline.describe(underspecified_grasp()))

### Layer 3 is the only place a world gets in

`SchemaOnlyContextProvider` adds nothing, which is the CI default and a useful baseline: whatever
the model answers, it answered from generic knowledge alone. A world-aware provider lives wherever
the world lives and is injected here -- this package never imports one.

In [ ]:
situated = LanguageModelBackend(
    client=ScriptedClient([]),
    context_provider=StaticContextProvider(
        "The target is a ceramic mug, 8 cm wide, with a handle on its right side, "
        "standing upright on a counter at 0.9 m. The left gripper is empty; the right "
        "gripper is holding a spoon."
    ),
    include_source=False,
)
print(situated.describe(underspecified_grasp()))

## 4. Resolving, offline

`ScriptedClient` replays fixed responses. This exercises decoding, write-back and validation --
everything except the model itself.

Write-back uses the same protocol as the probabilistic backend: bind each mapped variable by its
access path, push literals into the nested kwargs, then call the factory the caller supplied.
That factory is why this package never imports the classes it constructs.

In [ ]:
response = json.dumps([
    {
        "Grasp.approach_direction": "FRONT",
        "Grasp.vertical_alignment": "TOP",
        "Grasp.gripper": 0,
        "Grasp.rotate_gripper": False,
        "Grasp.manipulation_offset": 0.06,
    },
    {
        "Grasp.approach_direction": "TOP",
        "Grasp.vertical_alignment": "TOP",
        "Grasp.gripper": 1,
        "Grasp.rotate_gripper": True,
        "Grasp.manipulation_offset": 0.04,
    },
])

backend = LanguageModelBackend(client=ScriptedClient([response]), number_of_candidates=2)

for result in backend.evaluate(underspecified_grasp()):
    print(result)

Token `0` came back as the actual `Gripper` object from the extension -- identity round-tripped
through the token without the model ever seeing a Python object.

In [ ]:
result = next(iter(LanguageModelBackend(
    client=ScriptedClient([response]), number_of_candidates=2
).evaluate(underspecified_grasp())))

print(result.gripper is kitchen["left_gripper"])

## 5. The model is not a sound generator

So constraints are used **twice**: as prompt content to steer, and as a filter afterwards to
enforce. The filter is krrood's own selective evaluation, not a reimplementation.

Below, the model returns a negative offset in violation of the stated constraint, and it does
not survive.

In [ ]:
violating = json.dumps([{
    "Grasp.approach_direction": "FRONT",
    "Grasp.vertical_alignment": "TOP",
    "Grasp.gripper": 0,
    "Grasp.rotate_gripper": False,
    "Grasp.manipulation_offset": -0.02,
}])

try:
    list(LanguageModelBackend(
        client=ScriptedClient([violating]), number_of_candidates=1
    ).evaluate(underspecified_grasp()))
except NoValidCandidates as error:
    print("rejected:", error)

### Repair

A malformed or type-invalid answer is fed back with the reason attached, once per `max_repairs`.
Validation is against reflected metadata, so it costs nothing and catches most of what goes wrong.

In [ ]:
invalid = json.dumps([{
    "Grasp.approach_direction": "SIDEWAYS",
    "Grasp.vertical_alignment": "TOP",
    "Grasp.gripper": 0,
    "Grasp.rotate_gripper": False,
    "Grasp.manipulation_offset": 0.06,
}])
corrected = json.dumps([{
    "Grasp.approach_direction": "FRONT",
    "Grasp.vertical_alignment": "TOP",
    "Grasp.gripper": 0,
    "Grasp.rotate_gripper": False,
    "Grasp.manipulation_offset": 0.06,
}])

client = ScriptedClient([invalid, corrected])
backend = LanguageModelBackend(client=client, number_of_candidates=1, max_repairs=1)

print(next(iter(backend.evaluate(underspecified_grasp()))))
print("calls:", len(client.calls))
print()
print(client.calls[1][1].split("## Your previous answer was rejected")[1].strip()[:400])

In [ ]:
try:
    LanguageModelBackend(
        client=ScriptedClient([invalid, invalid]), number_of_candidates=1, max_repairs=1
    ).evaluate(underspecified_grasp()).__next__()
except RepairBudgetExhausted as error:
    print(error)

## 6. The line this backend will not cross

A language model can generate **values**. It cannot mint **individuals**: it can produce the
string `"the red cup"`, but not a live object with a pose and a place in the kinematic tree.
Asking it to would hand the constructor a hallucination.

So an individual-typed slot left as `...` is refused outright, with a message saying how to fix it.

In [ ]:
unminted = a(Grasp)(
    approach_direction=...,
    vertical_alignment=...,
    gripper=...,
    rotate_gripper=...,
    manipulation_offset=...,
)

try:
    list(LanguageModelBackend(client=ScriptedClient([])).evaluate(unminted))
except ReferentSlotHasNoDomain as error:
    print(error)

An empty extension is a different failure and gets a different exception. *"There is no such
thing here"* and *"nobody has looked yet"* demand opposite responses -- replan versus perceive --
so they must never collapse into one error.

In [ ]:
empty = a(Grasp)(
    approach_direction=...,
    vertical_alignment=...,
    gripper=variable(Gripper, []),
    rotate_gripper=...,
    manipulation_offset=...,
)

try:
    list(LanguageModelBackend(client=ScriptedClient([])).evaluate(empty))
except NoCandidateReferents as error:
    print(type(error).__name__, "->", error)

## 7. Provenance

Every exchange is kept. For any resolved object you can recover the exact prompt, the raw
response, which model produced it, and which attempt it was -- which is the thing the
probabilistic backend structurally cannot give you, since its only justification for a candidate
is a likelihood number.

In [ ]:
backend = LanguageModelBackend(client=ScriptedClient([invalid, corrected]), number_of_candidates=1)
list(backend.evaluate(underspecified_grasp()))

for exchange in backend.log:
    print(f"attempt {exchange.attempt} via {exchange.client_identity}: "
          f"{len(exchange.user)} chars in, {len(exchange.raw_response)} chars out")

## 8. A real model

Nothing above changes. Swap the client.

```bash
pip install anthropic
export ANTHROPIC_API_KEY=...
```

In [ ]:
# from llm_reasoner.clients.anthropic_client import AnthropicClient
#
# live = LanguageModelBackend(
#     client=AnthropicClient(model="claude-sonnet-4-5"),
#     context_provider=StaticContextProvider(
#         "The target is a ceramic mug, 8 cm wide, with a handle on its right side, "
#         "standing upright on a counter. The right gripper is holding a spoon."
#     ),
#     number_of_candidates=3,
# )
#
# for result in live.evaluate(underspecified_grasp()):
#     print(result)
#
# print(live.log[-1].raw)

Worth trying once it is wired up: give the situation above, which says the right gripper is
occupied and the mug has a handle on its right, and see whether the model picks the left gripper
and a side approach. That is generic knowledge the query does not contain and a fully-factorized
prior cannot supply -- and it is the only claim that justifies this package existing.

## What is deliberately missing

**Referents with no extension.** Right now `...` on an individual-typed slot raises. Phase 2 is
describe -> ground -> select: ask the model for a sortal description, compile it into EQL
conditions, evaluate them against the world, and only then select among the survivors. The LLM
supplies the intensional half of anchoring; EQL supplies the extensional half.

**Composition.** This is a terminal backend. A `ParameterResolver` seam of shape `Match -> Match`
would let the model resolve referents and hand the geometry to the probabilistic backend, which
would then treat the chosen object as conditioning evidence. The phases inside `_evaluate` are
already split along that line.

**Ambiguity.** When several candidates are equally good and the consequences differ, a coin flip
is the wrong answer. krrood's `Expert` abstraction is the existing escalation path.